# เปรียบเทียบการทำงานของ Deep Agent

โน้ตบุ๊กนี้สาธิตการสร้าง agent ด้วย Deep Agents SDK และเก็บข้อมูลการทำงานเพื่อใช้ประกอบการเปรียบเทียบ โดยครอบคลุมการทดสอบการตอบกลับ การเชื่อมต่อเครื่องมือค้นหา และการวัด latency กับ token usage

> หมายเหตุ: ผลลัพธ์ที่บันทึกไว้ในแต่ละรอบอาจขึ้นอยู่กับโมเดลและข้อมูล ณ เวลาที่รัน

<hr>


In [1]:
import os
import importlib.metadata

from deepagents import create_deep_agent
from langchain_core.messages import AIMessage

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing in the container."

MODEL = "openai:gpt-6-luna"

print("Deep Agents:", importlib.metadata.version("deepagents"))
print("OpenAI API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Model:", MODEL)

agent = create_deep_agent(
    model=MODEL,
    system_prompt="For the smoke test, reply with exactly: DEEP_AGENTS_OK",
)

result = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Run the smoke test."}
    ]
})

final_message = next(
    message
    for message in reversed(result["messages"])
    if isinstance(message, AIMessage)
)

print("Agent response:", final_message.content)
print("Usage:", final_message.usage_metadata)

Deep Agents: 0.7.19
OpenAI API key loaded: True
Model: openai:gpt-6-luna
Agent response: [{'type': 'text', 'text': 'DEEP_AGENTS_OK', 'annotations': [], 'id': 'msg_0c01e1cf32a8cf8a006abba15910c487d089ba0d02267a26bf', 'phase': 'final_answer'}]
Usage: {'input_tokens': 1988, 'output_tokens': 9, 'total_tokens': 1997, 'input_token_details': {'cache_creation': 1985, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


### ทดสอบการตอบกลับของ Agent

ส่วนนี้สร้าง agent ด้วยโมเดลที่กำหนดและส่งข้อความทดสอบสั้น ๆ จากนั้นอ่านคำตอบสุดท้ายและข้อมูลการใช้ token เพื่อตรวจสอบผลการเรียกใช้งาน

<hr>


In [2]:
import asyncio
from langchain_core.messages import AIMessage

print("Calling model...")

try:
    result = await asyncio.wait_for(
        agent.ainvoke({
            "messages": [
                {"role": "user", "content": "Run the smoke test."}
            ]
        }),
        timeout=120,
    )

    final_message = next(
        message
        for message in reversed(result["messages"])
        if isinstance(message, AIMessage)
    )

    print("Agent response:", final_message.content)
    print("Usage:", final_message.usage_metadata)

except Exception as exc:
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))

Calling model...
Agent response: [{'type': 'text', 'text': 'DEEP_AGENTS_OK', 'annotations': [], 'id': 'msg_015889d01357ea5f006abba1830b8887d0890c3878970dabe4', 'phase': 'final_answer'}]
Usage: {'input_tokens': 1988, 'output_tokens': 9, 'total_tokens': 1997, 'input_token_details': {'cache_creation': 0, 'cache_read': 1985}, 'output_token_details': {'reasoning': 0}}


### ทดสอบการเรียกซ้ำพร้อมกำหนดเวลาสูงสุด

ตัวอย่างนี้เรียก agent อีกครั้งโดยกำหนดเวลารอสูงสุด 120 วินาที หากใช้เวลาเกินกำหนดหรือเกิดข้อผิดพลาด จะแสดงชนิดและข้อความของข้อผิดพลาด เพื่อให้ตรวจสอบปัญหาการเรียกใช้งานได้สะดวก

<hr>


# เชื่อมต่อเครื่องมือค้นหาผ่าน Tavily MCP

หัวข้อนี้เตรียมการเชื่อมต่อกับ Tavily ผ่าน Model Context Protocol (MCP) และตรวจสอบรายการเครื่องมือที่ server เปิดให้ agent ใช้งาน โดยต้องมี API key ที่เกี่ยวข้องอยู่ใน environment

<hr>


In [1]:
import os

from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
from langchain.mcp import MCPAdapter

assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is missing."

transport = StreamableHttpTransport(
    url="https://mcp.tavily.com/mcp/",
    headers={
        "Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}",
    },
)

mcp_client = Client(transport)

async with MCPAdapter(mcp_client) as adapter:
    tavily_tools = await adapter.list_tools()

print("Connected Tavily tools:", [tool.name for tool in tavily_tools])

/tmp/ipykernel_18/61991752.py:5: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


Connected Tavily tools: ['tavily_search', 'tavily_extract', 'tavily_crawl', 'tavily_map', 'tavily_research']


### ให้ Agent ค้นหาเอกสาร Deep Agents

เมื่อนำเครื่องมือ Tavily มาให้ agent ใช้แล้ว จะส่งคำขอให้ค้นหาหน้าเอกสารทางการเกี่ยวกับ tools ผลลัพธ์จะแสดงคำตอบ รายชื่อเครื่องมือที่ถูกเรียก และข้อมูล token เพื่อช่วยดูว่า agent ใช้เครื่องมือค้นหาตามโจทย์หรือไม่

<hr>


In [2]:
from deepagents import create_deep_agent

MODEL = "openai:gpt-6-luna"

research_agent = create_deep_agent(
    model=MODEL,
    tools=tavily_tools,
    system_prompt=(
        "Use the provided Tavily tools to search the web. "
        "For factual claims, include the source title and URL. "
        "Do not invent sources."
    ),
)

response = await research_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use Tavily search to find the official Deep Agents "
                    "documentation page about tools. Return its page title "
                    "and URL, with no more than one source."
                ),
            }
        ]
    }
)

messages = response["messages"]
final_message = messages[-1]

print("Answer:")
print(final_message.content)

tool_calls = [
    call["name"]
    for message in messages
    for call in getattr(message, "tool_calls", [])
]
print("\nTools called:", tool_calls)
print("Usage:", getattr(final_message, "usage_metadata", None))

Answer:
[{'type': 'text', 'text': '**Tools - Docs by LangChain**  \nhttps://docs.langchain.com/oss/python/deepagents/tools', 'annotations': [], 'id': 'msg_035bade26ea71271006abba2fa5a1487d083d4f9ad73a81e12', 'phase': 'final_answer'}]

Tools called: ['tavily_search', 'tavily_search']
Usage: {'input_tokens': 7446, 'output_tokens': 26, 'total_tokens': 7472, 'input_token_details': {'cache_creation': 2128, 'cache_read': 5315}, 'output_token_details': {'reasoning': 0}}


# วัดผลกรณีโจทย์คณิตศาสตร์จาก Prompt ยาว

ส่วนนี้โหลดกรณีทดสอบ `long_prompt_math` จากไฟล์ benchmark แล้วสร้าง agent ที่ไม่มีเครื่องมือค้นเว็บ เพื่อให้ตอบจากข้อมูลที่โจทย์ให้เท่านั้น โค้ดจับเวลาการทำงาน รวบรวมจำนวนคำขอและ token ตลอดจนบันทึกคำตอบกับข้อมูลการรันเป็นไฟล์ JSON

### อ่านตารางผลการรันที่บันทึกไว้

ตารางถัดไปสรุปผลสามรอบ โดยแสดง latency และจำนวน input, output และ total tokens ส่วนแถว Median ใช้ดูค่ากลางของผลการรันเพื่อช่วยลดผลกระทบจากรอบที่เร็วหรือช้าผิดปกติ

<hr>


In [7]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

from deepagents import create_deep_agent

MODEL = "openai:gpt-6-luna"

SHARED_DIR = Path("/workspace/shared")

with (SHARED_DIR / "benchmark_cases.json").open(encoding="utf-8") as file:
    cases = json.load(file)

# with open("shared/benchmark_cases.json", encoding="utf-8") as file:
#     cases = json.load(file)

case = cases["long_prompt_math"]

math_agent = create_deep_agent(
    model=MODEL,
    tools=[],
    system_prompt=(
        "Follow every requirement in the user's request. "
        "Use only the supplied information. Do not use web search."
    ),
)

started_at = time.perf_counter()
response = await math_agent.ainvoke(
    {"messages": [{"role": "user", "content": case["prompt"]}]}
)
latency_s = time.perf_counter() - started_at

messages = response["messages"]
final_message = messages[-1]

content = final_message.content
if isinstance(content, list):
    answer = "\n".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )
else:
    answer = str(content)

assistant_messages = [
    message for message in messages
    if getattr(message, "type", None) == "ai"
]

usage = {
    "requests": sum(
        1 for message in assistant_messages
        if getattr(message, "usage_metadata", None)
    ),
    "input_tokens": sum(
        message.usage_metadata.get("input_tokens", 0)
        for message in assistant_messages
        if getattr(message, "usage_metadata", None)
    ),
    "output_tokens": sum(
        message.usage_metadata.get("output_tokens", 0)
        for message in assistant_messages
        if getattr(message, "usage_metadata", None)
    ),
    "total_tokens": sum(
        message.usage_metadata.get("total_tokens", 0)
        for message in assistant_messages
        if getattr(message, "usage_metadata", None)
    ),
}

run_id = datetime.now(timezone.utc).strftime(
    "deep-long-math-%Y%m%dT%H%M%SZ"
)

record = {
    "sdk": "deepagents",
    "case": "long_prompt_math",
    "run_id": run_id,
    "model": MODEL,
    "latency_s": round(latency_s, 2),
    "usage": usage,
    "requirements": case["requirements"],
    "answer": answer,
}

results_dir = SHARED_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / f"{run_id}.json"
output_path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Answer:\n", answer)
print("\nRun ID:", run_id)
print("Latency (seconds):", record["latency_s"])
print("Usage:", usage)
print("Saved:", output_path.resolve())

Answer:
 ## Executive summary

Using only the supplied test data, revenue grew **20%** from FY2024 to FY2025, and net income grew **27.5%**. Net margin increased from **8.0%** to **8.5%**. FY2024 net debt was **200 million THB**. FY2025 net debt is **unavailable** because cash was not reported.

## Financial table

_All figures below are supplied test data, not externally verified facts. Amounts are in million THB unless stated otherwise._

| Metric | FY2024 | FY2025 |
|---|---:|---:|
| Revenue | 1,000 | 1,200 |
| Net income | 80 | 102 |
| Total debt | 300 | 280 |
| Cash | 100 | Not reported |
| Revenue growth from FY2024 | — | 20% |
| Net income growth from FY2024 | — | 27.5% |
| Net margin | 8.0% | 8.5% |
| Net debt | 200 | Unavailable |

## Calculation notes

- **Revenue growth:**  
  \((\text{FY2025 revenue} - \text{FY2024 revenue}) \div \text{FY2024 revenue} \times 100\)  
  \(= (1{,}200 - 1{,}000) \div 1{,}000 \times 100 = 20\%\)

- **Net income growth:**  
  \((\text{FY2025 net 

# วัดผลกรณีค้นคว้าข้อมูลเกี่ยวกับ PTT

ตัวอย่างนี้โหลดกรณี `web_research_ptt` และเชื่อมต่อ Tavily MCP เพื่อให้ agent ค้นคว้าข้อมูลจากเว็บ โดยกำหนดแนวทางให้ให้ความสำคัญกับแหล่งข้อมูลทางการ ระบุ URL อ้างอิง และแยกตัวเลขที่รายงานจากตัวเลขที่คำนวณได้

ผลที่บันทึกประกอบด้วยคำตอบ เวลาใช้งาน จำนวน token รายชื่อเครื่องมือที่เรียก และข้อกำหนดของโจทย์ ซึ่งนำไปตรวจสอบและเปรียบเทียบกับกรณีทดสอบอื่นได้

<hr>


In [8]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
from langchain.mcp import MCPAdapter
from deepagents import create_deep_agent

MODEL = "openai:gpt-6-luna"
SHARED_DIR = Path("/workspace/shared")

with (SHARED_DIR / "benchmark_cases.json").open(encoding="utf-8") as file:
    cases = json.load(file)

case = cases["web_research_ptt"]

transport = StreamableHttpTransport(
    url="https://mcp.tavily.com/mcp/",
    headers={
        "Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}",
    },
)

async with MCPAdapter(Client(transport)) as adapter:
    tavily_tools = await adapter.list_tools()

    research_agent = create_deep_agent(
        model=MODEL,
        tools=tavily_tools,
        system_prompt=(
            "Research efficiently. Prioritize official PTT and SET sources. "
            "Stop when the requested facts are supported by sources. "
            "Cite a source URL for each factual claim. Distinguish reported "
            "figures from calculated figures. Do not guess."
        ),
    )

    started_at = time.perf_counter()
    response = await research_agent.ainvoke(
        {"messages": [{"role": "user", "content": case["prompt"]}]},
        config={"recursion_limit": 50},
    )
    latency_s = time.perf_counter() - started_at

messages = response["messages"]
final_message = messages[-1]

content = final_message.content
if isinstance(content, list):
    answer = "\n".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )
else:
    answer = str(content)

assistant_messages = [
    message for message in messages
    if getattr(message, "type", None) == "ai"
]

usage_messages = [
    message.usage_metadata
    for message in assistant_messages
    if getattr(message, "usage_metadata", None)
]

usage = {
    "requests": len(usage_messages),
    "input_tokens": sum(item.get("input_tokens", 0) for item in usage_messages),
    "output_tokens": sum(item.get("output_tokens", 0) for item in usage_messages),
    "total_tokens": sum(item.get("total_tokens", 0) for item in usage_messages),
}

tool_calls = [
    call.get("name")
    for message in assistant_messages
    for call in getattr(message, "tool_calls", [])
]

run_id = datetime.now(timezone.utc).strftime(
    "deep-ptt-research-%Y%m%dT%H%M%SZ"
)

record = {
    "sdk": "deepagents",
    "case": "web_research_ptt",
    "run_id": run_id,
    "model": MODEL,
    "mcp_server": "tavily",
    "latency_s": round(latency_s, 2),
    "usage": usage,
    "tool_calls": tool_calls,
    "requirements": case["requirements"],
    "answer": answer,
}

results_dir = SHARED_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / f"{run_id}.json"
output_path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Answer:\n", answer)
print("\nRun ID:", run_id)
print("Latency (seconds):", record["latency_s"])
print("Usage:", usage)
print("Tool calls:", tool_calls)
print("Saved:", output_path.resolve())

Answer:
 ## PTT Public Company Limited — as of 29 September 2026

### Identity
The Stock Exchange of Thailand (SET) identifies **PTT Public Company Limited** under the stock symbol **PTT**. ([SET company page](https://www.set.or.th/en/market/product/stock/quote/PTT/financial-statement/company-highlights))

### Most recently published completed fiscal year
**FY2025** is the latest completed annual reporting period shown in PTT’s investor-relations materials by the cutoff. PTT’s IR calendar lists its FY2025 results announcement on **19 February 2026**, and its IR site provides the FY2025 annual report and financial report. ([IR calendar](https://investor.pttplc.com/en/calendar); [FY2025 annual reports](https://investor.pttplc.com/en/downloads/one-report))

| FY2025 consolidated figure | Amount | Status and description |
|---|---:|---|
| Sales and service revenue | **THB 2,662,145 million** | **Reported**; the audited annual report labels this “Sales and service income.” ([FY2025 One Repo

### เพิ่มรายละเอียดการใช้ Cached Tokens ลงในผลลัพธ์

เซลล์นี้อ่านข้อมูล token usage ของข้อความจาก agent แล้วรวมจำนวน token ที่อ่านจาก cache และจำนวนที่ใช้สร้าง cache จากนั้นคำนวณ input tokens ที่ไม่อยู่ในสองกลุ่มดังกล่าว และเขียนค่าที่เพิ่มลงในไฟล์ผลลัพธ์ JSON ที่ระบุไว้

<hr>


In [9]:
import json

usage_messages = [
    message.usage_metadata
    for message in messages
    if getattr(message, "type", None) == "ai"
    and getattr(message, "usage_metadata", None)
]

cache_read = sum(
    item.get("input_token_details", {}).get("cache_read", 0)
    for item in usage_messages
)
cache_creation = sum(
    item.get("input_token_details", {}).get("cache_creation", 0)
    for item in usage_messages
)

path = SHARED_DIR / "results" / "deep-ptt-research-20260929T114601Z.json"
record = json.loads(path.read_text(encoding="utf-8"))

record["usage"]["cache_read_tokens"] = cache_read
record["usage"]["cache_creation_tokens"] = cache_creation
record["usage"]["uncached_input_tokens"] = max(
    record["usage"]["input_tokens"] - cache_read - cache_creation,
    0,
)

path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Updated:", path)
print(record["usage"])

Updated: /workspace/shared/results/deep-ptt-research-20260929T114601Z.json
{'requests': 7, 'input_tokens': 303992, 'output_tokens': 5685, 'total_tokens': 309677, 'cache_read_tokens': 224261, 'cache_creation_tokens': 79710, 'uncached_input_tokens': 21}


### คำนวณ Cached Tokens ในตัวแปร Usage

ส่วนนี้คำนวณค่า cache read, cache creation และ uncached input เช่นเดียวกัน แต่เก็บผลไว้ในตัวแปร `usage` ในหน่วยความจำ เพื่อให้สามารถนำไปใช้หรือบันทึกต่อในขั้นตอนอื่นได้

<hr>


In [10]:
cache_read = sum(
    item.get("input_token_details", {}).get("cache_read", 0)
    for item in usage_messages
)
cache_creation = sum(
    item.get("input_token_details", {}).get("cache_creation", 0)
    for item in usage_messages
)

usage["cache_read_tokens"] = cache_read
usage["cache_creation_tokens"] = cache_creation
usage["uncached_input_tokens"] = max(
    usage["input_tokens"] - cache_read - cache_creation,
    0,
)